In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import jscatter
from sklearn.decomposition import PCA
from umap import UMAP

In [2]:
%cd ..

f:\robusto\vqa_analysis


In [3]:
embedings_cache_path = "./external_qwen/uncleaned_embeddings_cache_keyed.pkl"
#load from pickle
import pickle
with open(embedings_cache_path, "rb") as f:
    embeddings_cache = pickle.load(f)

#loaded embeddings cache is a dict of text to embedding
print(f"Loaded embeddings cache with {len(embeddings_cache)} entries")

human_embedings = 0
vlm_embedings = 0
for key in embeddings_cache.keys():
    #key is a tuple of agent video question answer
    if "human" in  key[0]:
        human_embedings += 1
        #print(f"Human embedding key: {key}")
    else:
        vlm_embedings += 1
        #print(f"VLM embedding key: {key}")
        
        
print(f"Human embedings: {human_embedings}")
print(f"VLM embedings: {vlm_embedings}")
print(f"Total embedings: {len(embeddings_cache)}")

Loaded embeddings cache with 86000 entries
Human embedings: 6000
VLM embedings: 80000
Total embedings: 86000


In [4]:
print("Sample keys from embeddings cache:")
for i, key in enumerate(embeddings_cache.keys()):
    if i >= 5:
        break
    print(f"Key {i}: {key}")

Sample keys from embeddings cache:
Key 0: ('human_lima_1', 'Robusto2_153', 1, 1)
Key 1: ('human_lima_2', 'Robusto2_153', 1, 1)
Key 2: ('human_lima_3', 'Robusto2_153', 1, 1)
Key 3: ('human_lima_4', 'Robusto2_153', 1, 1)
Key 4: ('human_lima_5', 'Robusto2_153', 1, 1)


In [5]:
#load answers text cache
answers_text_cache_path = "./data/r2_clean.csv"
df_answers = pd.read_csv(answers_text_cache_path, keep_default_na=False) #keep_default_na=False to avoid converting empty strings to NaN
#df_answers = df_answers[df_answers["REPETITION"] == 1]
print(f"Loaded answers text cache with {len(df_answers)} entries")
df_answers.head()

Loaded answers text cache with 86000 entries


,AGENT,VIDEO,BLOCK,QUESTION_NUM,REPETITION,ANSWER
0,human_lima_1,Robusto2_153,1,1,1,The ego vehicle is accelerating slowly because...
1,human_lima_2,Robusto2_153,1,1,1,The ego vehicle is turning to the right
2,human_lima_3,Robusto2_153,1,1,1,the ego vehicle brakes and steers slightly to ...
3,human_lima_4,Robusto2_153,1,1,1,Braking to yield
4,human_lima_5,Robusto2_153,1,1,1,The ego vehicle is moving forward while mainta...


In [6]:
#first reduce all embedding cache using pca to 2 dimensions and store then into a df with agent video question answer and embedding
# --- Crear dataframe base ---
embeddings = []
for idx,row in df_answers.iterrows():
    key = (row["AGENT"], row["VIDEO"], row["QUESTION_NUM"], row["REPETITION"])
    if key in embeddings_cache:
        embedding = embeddings_cache[key]
        embeddings.append(embedding)
    else:
        raise ValueError(f"Embedding for key {key} not found in cache")

embeddings_arr = np.vstack(embeddings)
print(f"Embeddings array shape: {embeddings_arr.shape}")

coords_PCA = np.zeros((len(df_answers), 2))
coords_UMAP = np.zeros((len(df_answers), 2))

print("Starting dimensionality reduction by blocks...")
for block in [1, 2, 3, 4]:
    mask = df_answers["BLOCK"] == block
    print(f"Processing block {block} with {mask.sum()} entries")
    if mask.any():
        pca_block = PCA(n_components=2)
        umap_block = UMAP(n_components=2,metric="cosine",local_connectivity = 1, n_jobs=16)
        coords_PCA[mask.values] = pca_block.fit_transform(embeddings_arr[mask.values])
        coords_UMAP[mask.values] = umap_block.fit_transform(embeddings_arr[mask.values])

Embeddings array shape: (86000, 2560)
Starting dimensionality reduction by blocks...
Processing block 1 with 21500 entries


f:\robusto\vqa_analysis\venv\Lib\site-packages\umap\spectral.py:548: UserWarning: Spectral initialisation failed! The eigenvector solver
failed. This is likely due to too small an eigengap. Consider
adding some noise or jitter to your data.

Falling back to random initialisation!
  warn(
f:\robusto\vqa_analysis\venv\Lib\site-packages\umap\spectral.py:548: UserWarning: Spectral initialisation failed! The eigenvector solver
failed. This is likely due to too small an eigengap. Consider
adding some noise or jitter to your data.

Falling back to random initialisation!
  warn(


Processing block 2 with 21500 entries


f:\robusto\vqa_analysis\venv\Lib\site-packages\umap\spectral.py:548: UserWarning: Spectral initialisation failed! The eigenvector solver
failed. This is likely due to too small an eigengap. Consider
adding some noise or jitter to your data.

Falling back to random initialisation!
  warn(


Processing block 3 with 21500 entries
Processing block 4 with 21500 entries


f:\robusto\vqa_analysis\venv\Lib\site-packages\umap\spectral.py:548: UserWarning: Spectral initialisation failed! The eigenvector solver
failed. This is likely due to too small an eigengap. Consider
adding some noise or jitter to your data.

Falling back to random initialisation!
  warn(
f:\robusto\vqa_analysis\venv\Lib\site-packages\umap\spectral.py:548: UserWarning: Spectral initialisation failed! The eigenvector solver
failed. This is likely due to too small an eigengap. Consider
adding some noise or jitter to your data.

Falling back to random initialisation!
  warn(


In [7]:

interactive = df_answers.assign(pca_X=coords_PCA[:, 0], pca_Y=coords_PCA[:, 1])
interactive = interactive.assign(umap_X=coords_UMAP[:, 0], umap_Y=coords_UMAP[:, 1])
def get_group_color(agent):
    if "human" in agent:
        if "nyc" in agent:
            return "nyc"
        else:
            return "lima"
    else:
        return "vlm"
    
color_map = {
    "nyc": "#00E5FF", # Cian neón (sustituye al azul oscuro que no se ve)
    "lima": "#FF3D00", # Naranja-Rojo vibrante
    "vlm": "#00FF00", # Verde eléctrico (el que ya tienes, funciona bien)
}

interactive["group"] = interactive["AGENT"].map(get_group_color)
interactive.head()


,AGENT,VIDEO,BLOCK,QUESTION_NUM,REPETITION,ANSWER,pca_X,pca_Y,umap_X,umap_Y,group
0,human_lima_1,Robusto2_153,1,1,1,The ego vehicle is accelerating slowly because...,-0.290078,0.008543,6.446834,3.777449,lima
1,human_lima_2,Robusto2_153,1,1,1,The ego vehicle is turning to the right,-0.196138,0.112533,0.157007,4.162395,lima
2,human_lima_3,Robusto2_153,1,1,1,the ego vehicle brakes and steers slightly to ...,-0.215077,0.038999,0.783021,3.885792,lima
3,human_lima_4,Robusto2_153,1,1,1,Braking to yield,-0.076315,-0.064352,3.431466,3.833504,lima
4,human_lima_5,Robusto2_153,1,1,1,The ego vehicle is moving forward while mainta...,-0.163047,0.039195,1.478355,1.759962,lima


## Embeds by old pipeline - Uncleaned  - PCA reduced

In [12]:
import pandas as pd
import numpy as np

BLOCK_TO_PLOT = int(input("Enter the block number to plot (1-4): "))
df_block = interactive[interactive["BLOCK"] == BLOCK_TO_PLOT]

# Sample data
# Create an interactive scatter plot
scatterA = jscatter.Scatter(
    data= df_block,  # Filter for block 1
    x='pca_X',
    y='pca_Y',
    color_by='group',
    color_map=color_map,
    opacity=0.7,
    background_color="#111111",
    axes = True,
    title=f"PCA Scatter Plot for Block {BLOCK_TO_PLOT}",
    tooltip=True,
    tooltip_preview= "ANSWER",
    tooltip_preview_type="text",
    tooltip_properties=["pca_X","pca_Y","AGENT", "VIDEO", "QUESTION_NUM"],
    tooltip_histograms_size="large",
    tooltip_size="medium"
)
scatterA.axes(labels=['PCA 1', 'PCA 2'])

scatterB = jscatter.Scatter(
    data= df_block,  # Filter for block 1
    x='umap_X',
    y='umap_Y',
    color_by='group',
    color_map=color_map,
    opacity=0.7,
    background_color="#111111",
    axes = True,
    title=f"PCA Scatter Plot for Block {BLOCK_TO_PLOT}",
    tooltip=True,
    tooltip_preview= "ANSWER",
    tooltip_preview_type="text",
    tooltip_properties=["umap_X","umap_Y","AGENT", "VIDEO", "QUESTION_NUM"],
    tooltip_histograms_size="large",
    tooltip_size="medium"
)
scatterB.axes(labels=['UMAP 1', 'UMAP 2'])


print(f"Displaying scatter plot  of Block {BLOCK_TO_PLOT}...")
jscatter.link([scatterA],row_height=640)

Displaying scatter plot  of Block 2...


GridBox(children=(VBox(children=(HBox(children=(VBox(children=(<jscatter.widgets.button.Button object at 0x000…